This script is a one-time “format normalizer” for raw MEG inputs. It scans a source directory, finds MEG recordings that look like BIDS-ish runs (e.g. anything with '_meg' in the name), and ensures that, at the end, we have a flat directory of '.fif' files in a destination folder.

**If your original data are in CTF format (i.e. as '.ds' folders), this converts them to '.fif'.** If your original data are already .fif, it does nothing (and tells you to point your pipeline config directly at the existing files instead).

The original raw data remains untouched, but we force-convert to FIF format just to standardize input format for all internal pipeline operations.

### **OTHER NOTES:**

- Currently supports CTF or Elekta/Neuromag (MEGIN) systems, which should cover the vast majority of datasets.
- KIT / Yokogawa / Ricoh (CON/RAW format) currently **unsupported** (fairly rare anyways)

-------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path
import subprocess

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS

import pandas as pd
import mne
import shutil

### SET FILEPATHS:

original_data_path = Path(config['original_MEG_data_dir'])

destination_path = Path(config['MEG_data_directory'])
os.makedirs(destination_path, exist_ok=True)

Perform conversion:

- Skips conversion for any files already in FIF format
- If all FIF, skips conversion entirely (requires updating to config.yaml to point to original raw directory)
- If some files are FIFs but not all, converts non-FIFs and copies over FIFs
- Otherwise converts all files to FIF format

In [ ]:
# Collect MEG raw inputs (BIDS-ish) from the original dataset
ctf_inputs = []   # non-FIF CTF .ds
fif_inputs = []   # FIF MEG files

for root_directory, subdirectories, filenames in os.walk(original_data_path):
    root_path = Path(root_directory)

    # CTF-style .ds directories that look like MEG
    for subdirectory_name in subdirectories:
        subdirectory_path = root_path / subdirectory_name
        if subdirectory_path.suffix == ".ds" and "_meg" in subdirectory_path.name:
            ctf_inputs.append(subdirectory_path)

    # File-based MEG FIF
    for filename in filenames:
        file_path = root_path / filename

        # Skip obvious sidecars and non-MEG files
        if file_path.suffix in (".json", ".tsv", ".txt", ".jpg", ".png", ".pdf"):
            continue
        if "_meg" not in file_path.name:
            continue

        if file_path.suffix == ".fif":
            fif_inputs.append(file_path)

total_inputs = len(ctf_inputs) + len(fif_inputs)
print(f"Found {total_inputs} candidate MEG inputs: "
      f"{len(fif_inputs)} FIF, {len(ctf_inputs)} CTF (.ds).")

if total_inputs == 0:
    print("\nNo MEG-like inputs were found under:")
    print(f"   {original_data_path}")
    print("Nothing to convert.")
else:
    fif_fraction_percent = 100.0 * len(fif_inputs) / total_inputs
    print(f"Proportion already in FIF format: {fif_fraction_percent:.1f}%\n")

    # =========================================================
    # CASE 1: All files already FIF
    # =========================================================
    if len(fif_inputs) == total_inputs:
        print("All detected MEG inputs are already in FIF format.")
        print("No conversion or copying was performed.")
        print("\nThis script is intended to create FIF copies from non-FIF raw data.")
        print("Since everything is already FIF, you should point the "
              "'MEG_data_dir' field in your config.yaml")
        print("to the directory where these FIF files currently reside:")
        print(f"   {original_data_path}")
        print("\nOnce 'MEG_data_dir' is set appropriately, you can skip this step.")

    # =========================================================
    # CASE 2: Mixed FIF + CTF
    # =========================================================
    elif len(fif_inputs) > 0 and len(ctf_inputs) > 0:
        print("Mixed dataset detected: some files are FIF, some are CTF (.ds).")
        print("Actions:")
        print("  1) Convert all CTF (.ds) inputs to FIF")
        print("  2) Copy all existing FIF inputs into the destination")
        print("After this step, the destination directory will contain all inputs as FIF.")

        converted_paths = []
        failed_conversions = []
        skipped_existing_conversions = 0

        # -------------------------
        # Convert all CTF to FIF
        # -------------------------
        for ctf_dataset_path in ctf_inputs:
            basename = ctf_dataset_path.name  # e.g. sub-XXX_ses-YYY_task-rest_run-01_meg.ds

            # Strip .ds to get BIDS stem
            if not basename.endswith(".ds"):
                print(f"Warning: CTF dataset does not end with .ds, skipping: {basename}")
                continue
            bids_stem = basename[:-len(".ds")]  # e.g. sub-XXX_ses-YYY_task-rest_run-01_meg

            output_filename = bids_stem + ".fif"
            output_path = destination_path / output_filename

            if output_path.exists():
                print(f"Skipping conversion (already exists): {output_path}")
                skipped_existing_conversions += 1
                continue

            try:
                print(f"Converting CTF to FIF: {ctf_dataset_path} -> {output_path}")
                raw = mne.io.read_raw_ctf(str(ctf_dataset_path), preload=True, verbose="ERROR")
                raw.save(str(output_path), overwrite=True)
                converted_paths.append(output_path)
                print(f"Saved: {output_path}")
            except Exception as exception:
                print(f"Failed conversion for {ctf_dataset_path}")
                print(f"Error: {exception}")
                failed_conversions.append((ctf_dataset_path, str(exception)))

        # -------------------------
        # Copy all existing FIF into destination
        # -------------------------
        copied_fif_paths = []
        failed_copies = []
        skipped_existing_copies = 0

        for fif_input_path in fif_inputs:
            basename = fif_input_path.name  # e.g. sub-XXX_ses-YYY_task-rest_run-01_meg.fif

            if not basename.endswith(".fif"):
                print(f"Warning: FIF input does not end with .fif, skipping: {basename}")
                continue
            bids_stem = basename[:-len(".fif")]
            output_filename = bids_stem + ".fif"
            output_path = destination_path / output_filename

            if output_path.exists():
                print(f"Skipping copy (already exists): {output_path}")
                skipped_existing_copies += 1
                continue

            try:
                print(f"Copying FIF: {fif_input_path} -> {output_path}")
                shutil.copy2(str(fif_input_path), str(output_path))
                copied_fif_paths.append(output_path)
                print(f"Copied: {output_path}")
            except Exception as exception:
                print(f"Failed copy for {fif_input_path}")
                print(f"Error: {exception}")
                failed_copies.append((fif_input_path, str(exception)))

        # -------------------------
        # Summary (mixed)
        # -------------------------
        print("\n================== CONVERSION SUMMARY (MIXED DATASET) ==================")
        print(f"Converted CTF .ds -> FIF:            {len(converted_paths)}")
        print(f"Skipped conversions (dest exists):   {skipped_existing_conversions}")
        print(f"Failed conversions:                  {len(failed_conversions)}")
        print(f"Copied existing FIF inputs:          {len(copied_fif_paths)}")
        print(f"Skipped FIF copies (dest exists):    {skipped_existing_copies}")
        print(f"Failed FIF copies:                   {len(failed_copies)}")
        print(f"FIF proportion:                      {fif_fraction_percent:.1f}%")

        if failed_conversions:
            print("\nFailed conversions:")
            for input_path, error_message in failed_conversions:
                print("  ", input_path, "->", error_message)

        if failed_copies:
            print("\nFailed copies:")
            for input_path, error_message in failed_copies:
                print("  ", input_path, "->", error_message)

        print("\nAll FIF files should now be present directly in the destination directory:")
        print(f"   {destination_path}")
        print("You can now point 'MEG_data_dir' in config.yaml to this location.")

    # =========================================================
    # CASE 3: No FIF, only CTF
    # =========================================================
    elif len(fif_inputs) == 0 and len(ctf_inputs) > 0:
        print("No existing FIF files detected; all MEG inputs are CTF (.ds).")
        print("Proceeding to convert all CTF inputs to FIF.")

        converted_paths = []
        failed_conversions = []
        skipped_existing_conversions = 0

        for ctf_dataset_path in ctf_inputs:
            basename = ctf_dataset_path.name

            if not basename.endswith(".ds"):
                print(f"Warning: CTF dataset does not end with .ds, skipping: {basename}")
                continue
            bids_stem = basename[:-len(".ds")]
            output_filename = bids_stem + ".fif"
            output_path = destination_path / output_filename

            if output_path.exists():
                print(f"Skipping conversion (already exists): {output_path}")
                skipped_existing_conversions += 1
                continue

            try:
                print(f"Converting CTF to FIF: {ctf_dataset_path} -> {output_path}")
                raw = mne.io.read_raw_ctf(str(ctf_dataset_path), preload=True, verbose="ERROR")
                raw.save(str(output_path), overwrite=True)
                converted_paths.append(output_path)
                print(f"Saved: {output_path}")
            except Exception as exception:
                print(f"Failed conversion for {ctf_dataset_path}")
                print(f"Error: {exception}")
                failed_conversions.append((ctf_dataset_path, str(exception)))

        # Summary (CTF-only)
        print("\n================== CONVERSION SUMMARY (CTF-ONLY DATASET) ==================")
        print(f"Converted CTF .ds -> FIF:            {len(converted_paths)}")
        print(f"Skipped conversions (dest exists):   {skipped_existing_conversions}")
        print(f"Failed conversions:                  {len(failed_conversions)}")
        print(f"FIF proportion:                      {fif_fraction_percent:.1f}%")

        if failed_conversions:
            print("\nFailed conversions:")
            for input_path, error_message in failed_conversions:
                print("  ", input_path, "->", error_message)

        print("\nAfter this step, you can point 'MEG_data_dir' in config.yaml to:")
        print(f"   {destination_path}")

Final data-audit / validation:

In [ ]:
# ------------------------------------------------------------
# Post-conversion validation: detect missing FIF outputs
# ------------------------------------------------------------

original_bids_stems = set()

# 1. Collect all original BIDS stems from original_data_path
for root_directory, subdirectories, filenames in os.walk(original_data_path):
    root_path = Path(root_directory)

    # CTF-style .ds directories
    for subdirectory_name in subdirectories:
        subdirectory_path = root_path / subdirectory_name
        if subdirectory_path.suffix == '.ds' and '_meg' in subdirectory_path.name:
            name = subdirectory_path.name
            if name.endswith('.ds'):
                bids_stem = name[:-len('.ds')]
                original_bids_stems.add(bids_stem)

    # FIF inputs (in case of mixed datasets)
    for filename in filenames:
        file_path = root_path / filename
        if '_meg' in file_path.name and file_path.suffix == '.fif':
            name = file_path.name
            if name.endswith('.fif'):
                bids_stem = name[:-len('.fif')]
                original_bids_stems.add(bids_stem)


converted_bids_stems = set()

# 2. Collect all produced FIF stems from destination_path
for root_directory, subdirectories, filenames in os.walk(destination_path):
    root_path = Path(root_directory)
    for filename in filenames:
        file_path = root_path / filename
        if '_meg' in file_path.name and file_path.suffix == '.fif':
            name = file_path.name
            if name.endswith('.fif'):
                bids_stem = name[:-len('.fif')]
                converted_bids_stems.add(bids_stem)


# 3. Compare original vs converted
missing_bids_stems = sorted(list(original_bids_stems - converted_bids_stems))
extra_bids_stems = sorted(list(converted_bids_stems - original_bids_stems))

print("\n================== POST-CONVERSION VALIDATION ==================")
print(f"Original MEG runs detected:   {len(original_bids_stems)}")
print(f"FIF outputs detected:         {len(converted_bids_stems)}")
print(f"Missing FIF outputs:          {len(missing_bids_stems)}")
print(f"Unexpected extra FIF outputs: {len(extra_bids_stems)}")

if missing_bids_stems:
    print("\nRuns with NO corresponding FIF output:")
    for bids_stem in missing_bids_stems:
        print("   ", bids_stem)

if extra_bids_stems:
    print("\nFIF outputs not matched to any original run (unexpected):")
    for bids_stem in extra_bids_stems:
        print("   ", bids_stem)

print("\nValidation complete.\n")